# LightGBM con futuro · diario de experimentos (Pablo · A · rama `futuro`)

> ⚠️ **Solo existe en la rama `futuro`.** Usa información de **otros snapshots de la sesión** (pasado y futuro): rompe a propósito
> la regla 1 del equipo. Sus `oof/` y `preds/` **no deben llegar a `main`** (`stacking/stack.py` coge todo lo que hay en `oof/`).
> Es una línea de proyecto aparte para que Juanma pruebe la combinación con la red con futuro (`nn_future`, CV 0.887).

**Qué es:** `src/m_lgb_future.py`, un LightGBM LambdaRank (parámetros de referencia de `EQUIPO.md`) que puntúa cada tarea con
cuatro bloques de features de `src/features_future.py` (ninguna usa la etiqueta):

| Bloque | Qué contiene |
|---|---|
| `base` | columnas originales del snapshot |
| `logic` | reglas dentro del snapshot: plazo por tipo, factibilidad, ventana típica, EDF… (`logic_*`) |
| `past` | cuánto lleva la tarea en la lista, cambios respecto al snapshot anterior, racha de snapshots (`p_*`, `blk_*`, `gap_*`) |
| `future` | segundos hasta la siguiente instrucción al avión y su tipo, si encaja con la tarea, cuándo desaparece la tarea (`f_*`) |

**Cómo usar este notebook** (igual que `NN_pablo.ipynb`)
1. Ejecuta *Preparación* una vez (calcula las features, ~1–2 min).
2. Cada bloque: **hipótesis** → **variantes** (UNA cosa cambia respecto a la base) → **tabla** → conclusión.
3. Lo que gane se combina en `MEJOR`, se confirma con 3 semillas y se guarda para el stacking.

**Cómo leer las cifras:** `cv` = top-3 out-of-fold con `Data/folds.csv`. Pruebas rápidas con 1 semilla: diferencias < ~0.005
son ruido. `min_atco` = peor controlador (el privado es un solo controlador). Referencias: LightGBM limpio de Adri 0.648,
red con futuro 0.887, LightGBM con futuro antiguo (`CLAUDE.md`, folds viejos) 0.83.

**Qué no se repite de `LGB_adri.ipynb`** (solo se lee, no se toca): Adri ya probó, con datos limpios, sus grupos de features
(`snapshot`, `tipo`, `avion`, `tipo_estado`, `contexto`): sueltos aportan poco (+0.001 a +0.004) y **todos juntos empeoran**
(0.648 frente a 0.651). Eso es trabajo suyo y no se repite aquí. Sus **candidatos de parámetros** (`lento`, `regularizado`,
`xendcg`) están escritos pero no los ha lanzado: se prueban aquí con los datos con futuro (bloque 2), para que los dos
LightGBM se comparen con lo mismo.

## Preparación

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "cv"))
import m_lgb_future

data = m_lgb_future.build_data()      # una sola vez; todas las pruebas lo reutilizan
REF = {"LGB limpio (Adri)": 0.648, "red con futuro": 0.887}

In [ ]:
# Configuración base = parámetros de referencia de EQUIPO.md con los 4 bloques de features. Cada experimento cambia UNA clave.
BASE = dict(feats="base,logic,past,future", n_estimators=400, lr=0.05, leaves=31, min_child=50, subsample=0.8,
            colsample=0.7, trunc=5, reg_lambda=0.0, objective="lambdarank")


def probar(nombre, motivo, semillas=1, guardar=False, **cambios):
    """Lanza la CV con BASE + cambios; queda apuntado en experiments/lgb_future_runs.csv con su motivo.
    guardar=True genera oof/, preds/ y submissions/ (solo para la versión que va al stacking)."""
    cfg = {**BASE, **cambios}
    args = ["--name", nombre, "--tag", motivo] + (["--quick"] if not guardar else [])
    for k, v in cfg.items():
        args += [f"--{k.replace('_', '-')}", str(v)]
    a = m_lgb_future.parse(args)
    a.seeds = semillas
    return m_lgb_future.run(a, data)


def lanzar(bloque, variantes, semillas=1, sobre=None):
    """variantes: lista de (nombre, motivo, {cambios}). Siempre incluye la base para comparar en igualdad."""
    sobre = sobre or {}
    res = [("base", "referencia", probar(f"{bloque}/base", "referencia", semillas, **sobre))]
    for nombre, motivo, cambios in variantes:
        res.append((nombre, motivo, probar(f"{bloque}/{nombre}", motivo, semillas, **{**sobre, **cambios})))
    return tabla_bloque(res)


def tabla_bloque(res):
    t = pd.DataFrame(res, columns=["variante", "por qué", "cv"])
    ref = t.loc[t.variante == "base", "cv"].iloc[0]
    t["Δ vs base"] = t.cv - ref
    t["veredicto"] = t["Δ vs base"].map(lambda x: "—" if abs(x) < 1e-9 else ("mejora" if x > 0.005 else
                                                   ("empeora" if x < -0.005 else "ruido")))
    return (t.sort_values("cv", ascending=False).style
            .format({"cv": "{:.4f}", "Δ vs base": "{:+.4f}"})
            .background_gradient(subset=["Δ vs base"], cmap="RdYlGn", vmin=-0.02, vmax=0.02).hide(axis="index"))


def historial(filtro=""):
    """Todo lo probado (también desde la terminal), ordenado por CV."""
    r = pd.read_csv(m_lgb_future.LOG)
    r = r[r.name.str.contains(filtro, regex=False)] if filtro else r
    cols = ["when", "name", "tag", "cv", "min_atco", "seeds", "feats", "n_feats", "leaves", "min_child", "lr",
            "n_estimators", "colsample", "secs"]
    return r[[c for c in cols if c in r]].sort_values("cv", ascending=False).reset_index(drop=True)

## 0 · Punto de partida

Base con 3 semillas, generada desde la terminal (`python src/m_lgb_future.py`): es la versión `lgb_future` que ya está en
`oof/` y `preds/` para Juanma. La celda la mide con la métrica del equipo (`cv/metric.py`).

In [ ]:
from metric import report
report(pd.read_csv(ROOT / "oof" / "lgb_future.csv"), "lgb_future (base, 3 semillas)")

## 1 · ¿Qué bloque de información aporta?

**Hipótesis:** casi todo el salto viene del bloque `future` (la siguiente instrucción delata la tarea); `past` y `logic` aportan poco.
Quitar un bloque cada vez dice cuánto vale cada uno. Si quitar uno **no empeora**, sobra (menos ruido, más robustez).

| Variante | Qué se quita | Qué esperamos |
|---|---|---|
| sin `future` | f_* | caída grande (debería acercarse al LightGBM limpio + pasado) |
| sin `past` | p_*, blk_*, gap_* | caída pequeña |
| sin `logic` | logic_* | ruido: con futuro, las reglas a mano deberían sobrar |

In [ ]:
lanzar("feats", [
    ("sin_future", "¿cuánto aporta el futuro?",       dict(feats="base,logic,past")),
    ("sin_past",   "¿cuánto aporta el pasado?",        dict(feats="base,logic,future")),
    ("sin_logic",  "¿sobran las reglas a mano?",       dict(feats="base,past,future")),
])

**Conclusión bloque 1:** _(rellenar)_

## 2 · Parámetros (los candidatos de Adri, sobre datos con futuro)

**Hipótesis:** con features tan informativas el modelo se apoya en pocas columnas muy fuertes (`f_next_input`…); lo que
le obligue a repartir (menos columnas por árbol) o a ser más prudente puede generalizar mejor a controladores nuevos.

| Variante | Cambios (iguales que en `LGB_adri.ipynb`) | Por qué |
|---|---|---|
| `lento` | `lr=0.03`, `n_estimators=700` | aprender más despacio con más árboles: suele generalizar mejor |
| `regularizado` | `min_child=100`, `colsample=0.5`, `reg_lambda=5` | hojas más grandes, menos columnas por árbol, pesos más pequeños |
| `xendcg` | `objective="rank_xendcg"` | otra pérdida de ranking, más estable con etiquetas ruidosas |

~1 min por variante (1 semilla).

In [ ]:
lanzar("params", [
    ("lento",        "más lento con más árboles (Adri)",                 dict(lr=0.03, n_estimators=700)),
    ("regularizado", "hojas grandes, menos columnas, L2 (Adri)",         dict(min_child=100, colsample=0.5, reg_lambda=5.0)),
    ("xendcg",       "otra pérdida de ranking (Adri)",                   dict(objective="rank_xendcg")),
])

**Conclusión bloque 2:** _(rellenar)_

## 3 · Combinar lo que ha funcionado

Pon en `MEJOR` los cambios con veredicto **mejora** de los bloques 1 (p. ej. `feats=...` si sobra un bloque) y 2. Se comprueba con **3 semillas** frente a la base de 3 semillas
(sección 0), porque varios cambios que ayudan por separado no siempre suman.

In [ ]:
MEJOR = dict(
    # feats="base,past,future", lr=0.03, n_estimators=700, ...   <- rellenar con los ganadores de los bloques 1 y 2
)

In [ ]:
# Comprobación con 3 semillas. Se puede saltar: el bloque 4 solo necesita la celda de MEJOR.
cv_comb = probar("comb/" + ",".join(f"{k}={v}" for k, v in MEJOR.items()), "combinación de ganadores", semillas=3, **MEJOR)
print(f"combinación: {cv_comb:.4f}   referencias: {REF}")

**Conclusión bloque 3:** _(rellenar)_

## 4 · Versión para el stacking (rama `futuro`)

Si la combinación mejora a la base, genera la versión definitiva con 3 semillas en el formato común. Nombre nuevo
(`lgb_future_v2`…) para no pisar `lgb_future`. Juanma la combina con `nn_future` en esta rama.

In [ ]:
NOMBRE = "lgb_future_v2"
probar(NOMBRE, "versión stacking futuro: " + str(MEJOR), semillas=3, guardar=True, **MEJOR)

**Resultado:** _(rellenar)_

## 5 · Análisis de errores y comparación con la red con futuro

**Pregunta:** ¿fallan el LightGBM y la red en los mismos snapshots? Si no, la combinación de Juanma ganará. Se mira el acierto
por tipo de tarea de cada modelo y en cuántos snapshots acierta uno solo.

In [ ]:
from metric import hits, labels

TIPOS = ["entry_conflict", "exit_conflict", "entry_coord", "exit_coord", "assume", "transfer", "conflict",
         "non_conformance", "qos", "entry_conditions", "exit_conditions", "aircraft_request", "zone", "return_to_route"]


def comparar(modelos=("lgb_future", "nn_future")):
    lab = labels().set_index("sample_id")
    tipos = pd.concat([pd.read_csv(ROOT / "Data" / f"{f}.csv", usecols=["sample_id", "target_index", "task_index"])
                       for f in ("train", "validation")]).query("target_index == 1").set_index("sample_id").task_index
    h = pd.DataFrame({m: hits(pd.read_csv(ROOT / "oof" / f"{m}.csv")) for m in modelos})
    a, b = modelos
    print(f"top-3:  {a} {h[a].mean():.3f}   {b} {h[b].mean():.3f}")
    print(f"aciertan los dos {(h[a] & h[b]).mean():.3f} | solo {a} {(h[a] & ~h[b].astype(bool)).mean():.3f} | "
          f"solo {b} {(~h[a].astype(bool) & h[b]).mean():.3f} | ninguno {((h[a] == 0) & (h[b] == 0)).mean():.3f}")
    h["tipo"] = tipos.reindex(h.index).map(dict(enumerate(TIPOS)))
    t = h.groupby("tipo").agg(muestras=(a, "size"), **{a: (a, "mean"), b: (b, "mean")})
    t["Δ (lgb − red)"] = t[a] - t[b]
    return t.sort_values("muestras", ascending=False).style.format(
        {a: "{:.2f}", b: "{:.2f}", "Δ (lgb − red)": "{:+.2f}"}).background_gradient(subset=["Δ (lgb − red)"], cmap="RdYlGn")


comparar(("lgb_future", "nn_future"))      # cambiar a lgb_future_v2 si se genera

**Conclusión del análisis:** _(rellenar)_

## 6 · Siguientes ideas

- Probar aquí las features que está preparando Pau (exit_coordination, return_to_route) cuando estén.
- Si Juanma ve que la combinación `lgb_future` + `nn_future` gana, afinar el peso de cada uno en su `stack.py`.

## Historial completo

In [ ]:
historial()